In [3]:
import sqlite3
import pandas as pd

df = pd.read_csv(r"C:\econometrie2\projet S2\projetalleger.csv")
print(f"Fichier charge : {len(df):,} lignes")

conn = sqlite3.connect(r"C:\econometrie2\projet S2\nutrition.db")

df.to_sql("produits", conn, if_exists="replace", index=False)
print(f"Table produits creee : {len(df):,} lignes importees !")

C:\Users\remim\AppData\Local\Temp\ipykernel_23432\154315798.py:5: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"C:\econometrie2\projet S2\projetalleger.csv")


Fichier charge : 431,940 lignes
Table produits creee : 431,940 lignes importees !


## 1. Statistiques générales

In [4]:
query1 = """
SELECT 
    COUNT(*) as total_produits,
    ROUND(AVG(nutriscore_points), 2) as nutriscore_moyen,
    ROUND(AVG(nb_additifs), 2) as additifs_moyens,
    ROUND(AVG(calories), 2) as calories_moyennes
FROM produits
"""
pd.read_sql(query1, conn)

,total_produits,nutriscore_moyen,additifs_moyens,calories_moyennes
0,431940,11.22,1.58,272.93


## 2. Qualité nutritionnelle par catégorie

In [5]:
query2 = """
SELECT 
    groupe_alimentaire_1,
    COUNT(*) as nb_produits,
    ROUND(AVG(nutriscore_points), 2) as score_moyen,
    SUM(CASE WHEN nutriscore = 'A' THEN 1 ELSE 0 END) as nb_A,
    SUM(CASE WHEN nutriscore = 'E' THEN 1 ELSE 0 END) as nb_E
FROM produits
GROUP BY groupe_alimentaire_1
HAVING nb_produits >= 500
ORDER BY score_moyen ASC
"""
pd.read_sql(query2, conn)

,groupe_alimentaire_1,nb_produits,score_moyen,nb_A,nb_E
0,Fruits and vegetables,30303,2.78,14968,1920
1,Cereals and potatoes,46585,3.87,14948,2188
2,Beverages,30687,6.47,1972,8548
3,Composite foods,37658,6.64,3739,1787
4,Milk and dairy products,53324,10.41,2464,5367
5,Fish Meat Eggs,76997,11.20,16631,21354
6,Fat and sauces,31020,12.16,1376,10687
7,Salty snacks,32465,12.71,1912,9740
8,Sugary snacks,92858,20.76,763,61482


## 3. Top 10 meilleures marques

In [6]:
query3 = """
SELECT 
    marque,
    COUNT(*) as nb_produits,
    SUM(CASE WHEN nutriscore = 'A' THEN 1 ELSE 0 END) as nb_A,
    SUM(CASE WHEN nutriscore = 'B' THEN 1 ELSE 0 END) as nb_B,
    ROUND(AVG(nutriscore_points), 2) as score_moyen
FROM produits
WHERE marque IS NOT NULL
GROUP BY marque
HAVING nb_produits >= 50
ORDER BY score_moyen ASC
LIMIT 10
"""
pd.read_sql(query3, conn)

,marque,nb_produits,nb_A,nb_B,score_moyen
0,Sabarot,153,124,19,-6.63
1,Vivien Paille,110,63,37,-6.15
2,Volandrie,70,62,5,-4.17
3,Loué,218,204,4,-3.98
4,La Nouvelle Agriculture,1006,951,13,-3.97
5,"Marque Repère, Notre Jardin",227,175,37,-3.16
6,Golden Sun,58,19,34,-3.05
7,D'aucy,208,143,45,-2.73
8,Priméale,75,57,16,-2.63
9,Jean nicolas,56,48,6,-2.61


## 3. Transformation NOVA vs additifs

In [7]:
query4 = """
SELECT 
    transformation,
    COUNT(*) as nb_produits,
    ROUND(AVG(nb_additifs), 2) as additifs_moyens,
    ROUND(AVG(nutriscore_points), 2) as score_moyen
FROM produits
WHERE transformation IS NOT NULL
GROUP BY transformation
ORDER BY transformation ASC
"""
pd.read_sql(query4, conn)

,transformation,nb_produits,additifs_moyens,score_moyen
0,1.0,33879,0.06,-0.21
1,2.0,12400,0.01,10.01
2,3.0,58327,0.34,10.45
3,4.0,141907,2.69,13.60


## 4. Top 10 produits les plus scannés

In [9]:
query5 = """
SELECT 
    nom_produit,
    marque,
    nutriscore,
    transformation,
    nb_scans
FROM produits
WHERE nb_scans IS NOT NULL
AND nom_produit IS NOT NULL
ORDER BY nb_scans DESC
LIMIT 10
"""
pd.read_sql(query5, conn)

,nom_produit,marque,nutriscore,transformation,nb_scans
0,Fromage Blanc Nature,Milky Food Professional,A,3.0,3410.0
1,Sidi Ali,سيدي علي,A,NaN,2961.0
2,Eau minérale naturelle,sidi ali,A,1.0,2397.0
3,Sidi Ali,Sidi Ali,A,1.0,2206.0
4,Lait,Jaouda,B,3.0,1945.0
5,Jben,Jaouda,D,4.0,1883.0
6,Eau De Source,Cristaline,A,1.0,1814.0
7,Prince Goût Chocolat,LU,E,4.0,1744.0
8,Ain Saïss,Danone,A,1.0,1570.0
9,اكوافينا,AQUAFINA,A,NaN,1518.0
